# lab_22_capstone_lc_end_to_end

Goal
----
Thread the whole ISF phase-noise main spine in ONE pass for the ideal LC,
reusing the EXISTING common utilities (no re-implemented physics):

    Gamma(theta) = -sin(theta)        [isf_utils.gamma_lc_ideal]
      -> Gamma_rms = 1/sqrt(2)        [isf_utils.gamma_rms]
      -> S_phi(f) = Gamma_rms^2/q_max^2 * S_i/(2 pi f)^2     (1/f^2 skirt)
      -> Lorentzian D = Gamma_rms^2/(4 q_max^2) * S_i,  Df_3dB = D/pi  (v5 mapping)
      -> sigma_t  = integrate L(f) over a band   [noise_utils.integrate_rms_jitter]
      -> BER bathtub                              [serdes_utils.ber_bathtub]

This is the executable companion to
docs/03_isf_core_theory/capstone_lc_end_to_end.md. Every printed intermediate
carries a checkable `# -> expected value` comment so scripts/verify_examples.py
covers the entire chain when the same snippet is embedded in the doc.

Canonical numbers (AUTHORING_SPEC sec. 8 / 11.2):
    q_max = 1 pC,  S_i = 1e-24 A^2/Hz,  f0 = 5 GHz,  Gamma_rms^2 = 0.5 (true LC).

Convention note (factor of 2): we use the clean time-domain phase PSD
    S_phi = Gamma_rms^2/q_max^2 * S_i/(2 pi f)^2,   L(f) ~= 0.5 * S_phi,
matching capstone station 5's S_phi form and station 6's Lorentzian D. The
jitter integration in station 7 is driven, as in the doc, by the canonical
"datasheet" anchor L(1 MHz) = -100 dBc/Hz (1/f^2), so sigma_t = 447.9 fs.

Run
---
    PYTHONPATH=. python simulations/lab_22_capstone_lc_end_to_end.py

---

> 本 notebook 由 `scripts/make_notebooks.py` 從 `simulations/lab_22_capstone_lc_end_to_end.py` **自動產生**（generated snapshot，非手寫檔）。
> 權威版本是 repo 裡的 lab script；lab 更新後請重跑產生器同步。
> 執行需求：clone [isf-teaching-site](https://github.com/gmcycle7/isf-teaching-site)（要 import `simulations/common`）＋ `numpy` / `scipy` / `matplotlib`。

In [ ]:
# --- Setup：本 notebook 需要教學網站 repo 的 simulations/common 模組 ---
# 還沒有原始碼的話，先 clone repo，並把本 notebook 放在 repo 目錄樹內執行：
#     git clone https://github.com/gmcycle7/isf-teaching-site.git
# 相依套件只有三個：pip install numpy scipy matplotlib（外加 jupyter 本身）
import sys
from pathlib import Path

def _find_repo_root():
    """從目前工作目錄往上找，直到看到 simulations/common 為止。"""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "simulations" / "common").is_dir():
            return base
    raise FileNotFoundError(
        "找不到 simulations/common —— 請把本 notebook 放進 isf-teaching-site "
        "repo 目錄樹內執行（git clone https://github.com/gmcycle7/isf-teaching-site.git），"
        "或手動把 <repo>/simulations/common 加入 sys.path")

ROOT = _find_repo_root()
for _p in (str(ROOT), str(ROOT / "simulations" / "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root:", ROOT)

# CJK 字型：圖的標籤有繁體中文；找不到 CJK 字型只影響文字顯示、不影響任何數值
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm
_avail = {f.name for f in _fm.fontManager.ttflist}
_cjk = next((f for f in ["Heiti TC", "Arial Unicode MS", "STHeiti",
                         "Hiragino Sans GB", "Songti SC", "PingFang TC",
                         "Noto Sans CJK TC", "Microsoft JhengHei"]
             if f in _avail), None)
if _cjk:
    plt.rcParams["font.family"] = [_cjk, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False   # ASCII 減號，避免變方塊
print("CJK font:", _cjk or "(none found — 中文標籤可能顯示為方塊)")

In [ ]:
import numpy as np

from isf_utils import gamma_lc_ideal, gamma_rms
from noise_utils import integrate_rms_jitter, leeson_one_over_f2
from serdes_utils import ber_bathtub

In [ ]:
def main():
    print("[lab_22] ideal-LC capstone: Gamma -> Gamma_rms -> S_phi -> "
          "Lorentzian -> sigma_t -> BER")

    # ---- canonical inputs -------------------------------------------------
    q_max = 1e-12          # node max charge swing [C]  (= 1 pC)
    S_i = 1e-24            # one-sided white current PSD [A^2/Hz]
    f0 = 5e9               # oscillation frequency [Hz] (5 GHz)

    # ---- station 3+4: Gamma(theta) = -sin(theta) -> Gamma_rms -------------
    theta = np.linspace(0.0, 2 * np.pi, 4001, endpoint=True)
    gamma = gamma_lc_ideal(theta)                 # = -sin(theta)
    Grms = gamma_rms(theta, gamma)                # = 1/sqrt(2)
    Grms2 = Grms ** 2
    print("Gamma_rms      =", round(float(Grms), 4), "   # -> 0.7071")
    print("Gamma_rms^2    =", round(float(Grms2), 4), "   # -> 0.5")

    # ---- station 5: phase PSD S_phi(f) (clean time-domain 1/f^2 skirt) ----
    # S_phi(f) = Gamma_rms^2/q_max^2 * S_i/(2 pi f)^2
    df = 1e6               # evaluate the skirt at 1 MHz offset
    S_phi_1MHz = Grms2 / q_max ** 2 * S_i / (2 * np.pi * df) ** 2
    print("S_phi(1MHz)    =", "{:.4e}".format(S_phi_1MHz),
          "rad^2/Hz   # -> 1.2665e-14")
    # corresponding SSB L(f) ~= 0.5*S_phi  (time-domain clean convention).
    # This clean /2 form sits 3 dB above the [P1] Eq.(21) /4 SSB form
    # (capstone station 5 reports -145 dBc/Hz via the /4 convention).
    L_1MHz = 10 * np.log10(0.5 * S_phi_1MHz)
    print("L(1MHz) clean  =", round(float(L_1MHz), 2),
          "dBc/Hz   # -> -141.98")

    # ---- station 6: Lorentzian diffusion D and 3-dB linewidth -------------
    # D = Gamma_rms^2/(4 q_max^2) * S_i ;  Df_3dB = D/pi  (v5: kappa^2/2)
    D = Grms2 / (4 * q_max ** 2) * S_i
    df_3dB = D / np.pi
    print("D (diffusion)  =", round(float(D), 4),
          "rad^2/s   # -> 0.125")
    print("Df_3dB (FWHM)  =", round(float(df_3dB), 4),
          "Hz   # -> 0.0398")

    # ---- station 7: integrate L(f) over a band -> rms jitter --------------
    # Use the canonical datasheet anchor (example C): L(1MHz)=-100 dBc/Hz,
    # 1/f^2 skirt, integrate 1 MHz -> 100 MHz, via the SHARED integrator.
    f = np.logspace(6, 8, 20001)                  # 1 MHz .. 100 MHz
    L_band = leeson_one_over_f2(f, L_ref_dbc=-100.0, f_ref=1e6)
    sigma_t, sigma_phi = integrate_rms_jitter(f, L_band, f0=f0,
                                              fmin=1e6, fmax=100e6)
    print("sigma_phi      =", "{:.4e}".format(sigma_phi),
          "rad   # -> 1.407e-02")
    print("sigma_t        =", "{:.4e}".format(sigma_t),
          "s   # -> 4.479e-13")
    print("sigma_t [fs]   =", round(float(sigma_t * 1e15), 1),
          "fs   # -> 447.9")

    # ---- station 8: sigma_t -> BER bathtub --------------------------------
    ui = 100e-12           # 10 Gb/s -> UI = 100 ps
    # BER at the eye center (t = 0) and the floor it sets
    ber_center = float(ber_bathtub(np.array([0.0]), sigma_t, ui)[0])
    # at center: BER = Q(UI/2/sigma_t); UI/2/sigma_t = 50ps/447.9fs ~ 111.6
    # -> astronomically small; report -log10 so the chain stays checkable.
    print("UI/2 / sigma_t =", round(float((ui / 2) / sigma_t), 1),
          "   # -> 111.6")
    # RJ peak-to-peak budget for BER=1e-12: Q^-1(1e-12)~7.03, pp=2*7.03*sigma_t
    rj_pp = 2 * 7.03 * sigma_t
    eye_closure = rj_pp / ui
    print("RJ_pp [ps]     =", round(float(rj_pp * 1e12), 3),
          "ps   # -> 6.297")
    print("eye closure    =", round(float(eye_closure * 100), 2),
          "% UI   # -> 6.3")
    # sanity: center BER is effectively zero for this tiny sigma_t
    print("BER(center)    =", "{:.1e}".format(ber_center),
          "   # -> 1.0e-300")

    print("[lab_22] done: state -> ISF -> S_phi -> linewidth -> "
          "jitter -> BER, one pass.")

In [ ]:
# 執行整個 lab（對應原 script 的 __main__）
main()